<a href="https://colab.research.google.com/github/juliawol/WB_Sufficiency/blob/main/Notebooks/WB_Query.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 16.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [1]:
!pip install --upgrade chromadb


In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [2]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")


In [3]:
client = chromadb.EphemeralClient()


In [4]:
data = pd.read_csv("/content/qa_dataset_labeled.csv")

In [5]:
# Drop duplicate IDs from the dataset
data = data.drop_duplicates(subset="NmId")


In [6]:

# Load tokenizer and fine-tuned model
model_path = "/content/fine_tuned_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model =  AutoModelForSequenceClassification.from_pretrained(model_path)

# Function to generate embeddings
def generate_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        embedding = outputs.hidden_states[-1][:, 0, :].squeeze().numpy()  # CLS token embedding
    return embedding

In [9]:
collection = client.get_or_create_collection(name="my_collection")


In [13]:

id_to_description = {row["NmId"]: row["Description"] for _, row in data.iterrows()}

for _, row in data.iterrows():
    embedding = generate_embedding(row["Description"])
    collection.add(
        embeddings=[embedding],
        metadatas=[{"id": str(row["NmId"]), "description": row["Description"]}],
        ids=[str(row["NmId"])]
    )
print("Data added to ChromaDB.")


Data added to ChromaDB.


In [14]:
def retrieve_description_by_id(product_id, collection):
    # Query ChromaDB to retrieve metadata directly by ID
    results = collection.get(ids=[str(product_id)])

    if not results or "metadatas" not in results or len(results["metadatas"]) == 0:
        return None

    # Extract the description from the metadata
    description = results["metadatas"][0].get("description")
    return description


In [15]:
def inference_pipeline(product_id, question, collection, model, tokenizer):
    # Retrieve the product description by ID
    description = retrieve_description_by_id(product_id, collection)

    if not description:
        return {"error": "No description found for the provided product ID."}

    # Tokenize the question and description
    inputs = tokenizer(question, description, return_tensors="pt", truncation=True, padding=True, max_length=512)

    # Predict using the fine-tuned model
    outputs = model(**inputs)
    probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
    label = torch.argmax(probabilities).item()
    confidence = probabilities[0][label].item()

    return {"label": label, "confidence": confidence, "description": description}


In [16]:
product_id = int(input("Enter Product ID: "))
question = input("Enter Question: ")

result = inference_pipeline(product_id, question, collection, model, tokenizer)
if "error" in result:
    print(result["error"])
else:
    print(f"Label: {result['label']} (Confidence: {result['confidence']:.2f})")
    print(f"Description: {result['description']}")

Enter Product ID: 3440033
Enter Question: Откуда такая цена?
Label: 0 (Confidence: 0.88)
Description: {'Страна производства': 'Таиланд', 'ТНВЭД': '6403511500', 'Метод крепления подошвы': 'Вулканизация', 'Высота подошвы': '1 см', 'Коллекция': 'Осень-Зима 2024', 'Длина упаковки': '34 см', 'Ортопедия': 'нет', 'Материал стельки': 'Искусственная шерсть', 'Ставка НДС': '20', 'Материал подошвы обуви': 'Полиуретан', 'Высота обуви': 'высокие', 'Материал подкладки обуви': 'Искусственная шерсть', 'Дата окончания действия сертификата/декларации': '25.05.2025', 'Высота упаковки': '13 см', 'Назначение обуви': 'повседневная', 'Цвет': 'черный', 'Особенности модели': 'Gore-Tex мембрана', 'Модель ботинок': 'ботинки', 'Дата регистрации сертификата/декларации': '26.05.2022', 'Декоративные элементы': 'без элементов', 'Номер сертификата соответствия': 'ЕАЭС RU С-DK.АЯ46.В.25234/22', 'Комплектация': 'Ботинки - 1 пара', 'Вид застежки': 'Шнурки', 'Полнота обуви (EUR)': 'F (6)', 'Ширина упаковки': '34 см', 'Пол

Вопрос про цену - один из самых часто встречающихся. На бейзлайне модель не ьыла уверена, к чему относитьь такие вопросы, так как слово "цена" фигурирует в карточках довольно часто. Намеренно включили в обучающую выборку такие моменты.

In [17]:
product_id = int(input("Enter Product ID: "))
question = input("Enter Question: ")

print(f"Label: {result['label']} (Confidence: {result['confidence']:.2f})")
print(f"Description: {result['description']}")

Enter Product ID: 2025564
Enter Question: Состав?
Label: 0 (Confidence: 0.88)
Description: {'Страна производства': 'Таиланд', 'ТНВЭД': '6403511500', 'Метод крепления подошвы': 'Вулканизация', 'Высота подошвы': '1 см', 'Коллекция': 'Осень-Зима 2024', 'Длина упаковки': '34 см', 'Ортопедия': 'нет', 'Материал стельки': 'Искусственная шерсть', 'Ставка НДС': '20', 'Материал подошвы обуви': 'Полиуретан', 'Высота обуви': 'высокие', 'Материал подкладки обуви': 'Искусственная шерсть', 'Дата окончания действия сертификата/декларации': '25.05.2025', 'Высота упаковки': '13 см', 'Назначение обуви': 'повседневная', 'Цвет': 'черный', 'Особенности модели': 'Gore-Tex мембрана', 'Модель ботинок': 'ботинки', 'Дата регистрации сертификата/декларации': '26.05.2022', 'Декоративные элементы': 'без элементов', 'Номер сертификата соответствия': 'ЕАЭС RU С-DK.АЯ46.В.25234/22', 'Комплектация': 'Ботинки - 1 пара', 'Вид застежки': 'Шнурки', 'Полнота обуви (EUR)': 'F (6)', 'Ширина упаковки': '34 см', 'Пол': 'Женский

В данной ситуации модель учла, что состав формально описан в карточке, но по факту там его нет (указана только вода). Обучение произошло в результате часто встречающихся вопросов такого рода в обучающем сете.

In [13]:
product_id = int(input("Enter Product ID: "))
question = input("Enter Question: ")

result = inference_pipeline(product_id, question, collection, model, tokenizer)

print(f"Label: {result['label']} (Confidence: {result['confidence']:.2f})")
print(f"Description: {result['description']}")

Enter Product ID: 2025407
Enter Question: А когда новая поставка?
Label: 0 (Confidence: 0.88)
Description: {'Упаковка': 'коробка', 'Комплектация': "краска-уход для волос L'Oreal Paris (Лореаль Париж) - 1 шт", 'Прямые поставки от производителя': 'да', 'Номер декларации соответствия': 'ЕАЭС N RU Д-BE.РА01.В.97110/21', 'Форма упаковки': 'без давления', 'Срок годности': '36 месяцев', 'Длина упаковки': '8 см', 'Дата регистрации сертификата/декларации': '28.05.2021', 'Раздел меню': 'Окрашивание волос и химическая завивка', 'Объем товара': '180 мл', 'Тип краски': 'полустойкая', 'Ширина упаковки': '9 см', 'Тон краски для волос': '1021, Светло-светло-русый перламутровый', 'Особенности краски для волос': 'безаммиачная', 'Высота упаковки': '17 см', 'Страна производства': 'Бельгия', 'Цвет': 'светло-русый', 'Состав': 'вода', 'Вес товара с упаковкой (г)': '239 г', 'Назначение косметического средства': 'для волос', 'Дата окончания действия сертификата/декларации': '27.05.2026'}  Краска для волос Cast

Изначально, до дообучения, модель была склонна относить вопросы о поставках к классу 1. В обучающую выборку включили множество вопросов в разных формулировках на такцю тему, отсюда довольно высокая уверенность.